In [ ]:
!pip install torch torchvision torchaudio
!pip install tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
print(f"Success! Torch version: {torch.__version__}")
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.preprocessing import MinMaxScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class HouseDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform
        self.features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition',
                         'floors','waterfront','view','sqft_above','sqft_basement','yr_built',
                         'yr_renovated','zipcode','sqft_living15','sqft_lot15']

        raw_prices = self.df['price'].values.astype(np.float32)
        self.min_price = raw_prices.min()
        self.max_price = raw_prices.max()

        self.target = (raw_prices - self.min_price) / (self.max_price - self.min_price)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        tab_data = self.df.iloc[idx][self.features].values.astype(np.float32)

        img_id = self.df.iloc[idx]['id']
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = self.target[idx]

        return torch.tensor(tab_data), image, torch.tensor(label)

In [ ]:
from torch.nn import functional as F

def get_gradcam(model, img_tensor, tab_tensor):

    model.eval()

    features = []
    def hook_feature(module, input, output):
        features.append(output)

    handle = model.cnn.layer4.register_forward_hook(hook_feature)

    output = model(tab_tensor, img_tensor)

    output.backward()

    handle.remove()
    return heatmap

In [ ]:
df = pd.read_excel('/content/sample_data/train(1).xlsx')
df1 = pd.read_excel('/content/sample_data/test2.xlsx')
numeric_cols = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition','floors','waterfront','view','sqft_above','sqft_basement','yr_built','yr_renovated','zipcode','sqft_living15','sqft_lot15']

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())

scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
# Taking only 15k images to train
df_sub = df.iloc[:15000].copy()
print(f"Numeric columns defined: {numeric_cols}")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df_sub, test_size=0.2, random_state=42)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(HouseDataset(train_df, r"/content/drive/MyDrive/images/", transform=transform), batch_size=32, shuffle=True)
val_loader = DataLoader(HouseDataset(val_df, r"/content/drive/MyDrive/images/", transform=transform), batch_size=32, shuffle=False)